# LLaMA_Factory Installation 


In [1]:

%rm -rf LLaMA-Factory

!git clone https://github.com/hiyouga/LLaMA-Factory.git

%cd LLaMA-Factory

!git checkout d325a1a7c7052946bd3b1e666459222c06ae5cb5

%ls

!pip install -e .[torch,bitsandbytes]


Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 24425, done.
remote: Total 24425 (delta 0), reused 0 (delta 0), pack-reused 24425 (from 1)
Receiving objects: 100% (24425/24425), 12.06 MiB | 25.41 MiB/s, done.
Resolving deltas: 100% (17622/17622), done.
/kaggle/working/LLaMA-Factory
fatal: reference is not a tree: d325a1a7c7052946bd3b1e666459222c06ae5cb5
assets/       docker/    Makefile        README.md         scripts/  tests/
CITATION.cff  examples/  MANIFEST.in     README_zh.md      setup.py  tests_v1/
data/         LICENSE    pyproject.toml  requirements.txt  src/
Obtaining file:///kaggle/working/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 28.0 MB/s eta 0:00:0

# Login to HF

In [ ]:
hf_token = 'your_huggingface_token_here'
!huggingface-cli login --token {hf_token}


⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `lama` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `lama`


# Dataset Preparation
- add `system` Prompt
- add `instruction` - User Prompt
- add `input` - Text
- add `output` - Ground-Truth for Generation

In [3]:
import json
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# same data with summaries 
file_path  = "hf://datasets/vishnupriyavr/wiki-movie-plots-with-summaries/wiki_movie_plots_deduped_with_summaries.csv"

df = pd.read_csv(file_path)
df.head(2)

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot,PlotSummary
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr...",Carrie Nation and her followers burst into a s...
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov...","The moon, painted with a smiling face hangs ov..."


In [ ]:
# reanme to match expected column names in llama-factory
df = df.rename(columns={"Plot": "input"})

df.head(2)

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,input,PlotSummary
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr...",Carrie Nation and her followers burst into a s...
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov...","The moon, painted with a smiling face hangs ov..."


#  `Summarization` taks

In [7]:
SYSTEM_PROMPT = "You are an expert detailed summarizer, Generate non-spoiler summaries."
INSTRUCTION_PROMPT = "Generate non-spoiler summary for the following text:\n"

In [ ]:
df["system"] = SYSTEM_PROMPT
df["instruction"] = INSTRUCTION_PROMPT

# Rename  
df = df.rename(columns={"PlotSummary": "output"})

df = df[["input", "output", "system", "instruction"]]

df.head(2)

,input,output,system,instruction
0,"A bartender is working at a saloon, serving dr...",Carrie Nation and her followers burst into a s...,"You are an expert detailed summarizer, Generat...",Generate non-spoiler summary for the following...
1,"The moon, painted with a smiling face hangs ov...","The moon, painted with a smiling face hangs ov...","You are an expert detailed summarizer, Generat...",Generate non-spoiler summary for the following...


# Splitting Data Train/Test

In [9]:
df.shape

(34886, 4)

In [10]:
# Split data
train_data, test_data = train_test_split(df, train_size=24000/34886, random_state=42)
train_data.shape, test_data.shape

((24000, 4), (10886, 4))

In [11]:
# To save a pandas DataFrame as a .jsonl (JSON Lines) file
TRAIN_FILE_PATH = "/kaggle/working/LLaMA-Factory/train.jsonl"
TEST_FILE_PATH = "/kaggle/working/LLaMA-Factory/test.jsonl"

train_data.to_json(TRAIN_FILE_PATH, orient="records", lines=True, force_ascii=False)
test_data.to_json(TEST_FILE_PATH, orient="records", lines=True, force_ascii=False)

 # Configure LLaMA-Factory for the new datasets


In [12]:
import os

dataset_info_path = "./data/dataset_info.json"

In [13]:
custom_dataset_entry = {
    "custom_finetune_train": {
        "file_name": TRAIN_FILE_PATH,
        "columns": {
            "system": "system",  
            "prompt": "instruction",
            "response": "output",
            "query": "input"    

        }
    },
    "custom_finetune_test": {
        "file_name": TEST_FILE_PATH,
        "columns": {
            "system": "system",    
            "prompt": "instruction", 
            "response": "output",
            "query": "input"        
        }
    }
}


In [14]:
if os.path.exists(dataset_info_path):
    with open(dataset_info_path, "r") as f:
        dataset_info = json.load(f)


# Add or update the custom dataset entry (extraction)
dataset_info.update(custom_dataset_entry)

# Write back the updated info
with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)


In [15]:
!tail -20 ./data/dataset_info.json

  },
  "custom_finetune_train": {
    "file_name": "/kaggle/working/LLaMA-Factory/train.jsonl",
    "columns": {
      "system": "system",
      "prompt": "instruction",
      "response": "output",
      "query": "input"
    }
  },
  "custom_finetune_test": {
    "file_name": "/kaggle/working/LLaMA-Factory/test.jsonl",
    "columns": {
      "system": "system",
      "prompt": "instruction",
      "response": "output",
      "query": "input"
    }
  }
}

## - Create `Training Configuration` File


In [ ]:
%%writefile ./examples/train_lora/summarization_finetune_gemma3.yaml

### model
model_name_or_path: google/gemma-3-270m-it    
trust_remote_code: true                  

### method
stage: sft                                
do_train: true                         
finetuning_type: lora                     
lora_rank: 64                            
lora_target: all                         

### dataset
dataset: custom_finetune_train     
eval_dataset: custom_finetune_test  
template: gemma                    
cutoff_len: 4096                         
max_samples: 10000                         
overwrite_cache: true                    
preprocessing_num_workers: 16          

### output
output_dir: ../finetuned_models/gemma3   
logging_steps: 10                      
save_steps: 500                          
plot_loss: false                         

### train
per_device_train_batch_size: 1      
gradient_accumulation_steps:  8        
learning_rate: 5.0e-5             
num_train_epochs: 1.0                   
lr_scheduler_type: cosine               
warmup_ratio: 0.1                       
bf16: true                               
ddp_timeout: 180000000                   

### eval
# val_size: 0.1                       
per_device_eval_batch_size: 1     
eval_strategy: steps                     
eval_steps: 100                          

report_to: none



Overwriting ./examples/train_lora/summarization_finetune_gemma3.yaml


 # Start Training

In [23]:
!llamafactory-cli train ./examples/train_lora/summarization_finetune_gemma3.yaml 

2025-11-14 22:34:00.773808: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763159640.799736    1393 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763159640.807995    1393 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[INFO|2025-11-14 22:34:05] llamafactory.launcher:143 >> Initializing 2 distributed tasks at: 127.0.0.1:42575
W1114 22:34:07.076000 1406 torch/distributed/run.py:792] 
W1114 22:34:07.076000 1406 torch/distributed/run.py:792] *****************************************
W1114 22:34:07.076000 1406 torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overl

# Inference 

In [24]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from pprint import pprint

In [25]:
def generate_response(messages, max_new_tokens=1024):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        top_k=None,
        temperature=None,
        top_p=None,
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

In [37]:
base_model_id = "google/gemma-3-270m-it"
adapter_model_id = "/kaggle/working/finetuned_models/gemma3"

In [32]:
system = test_data.iloc[0]["system"]
input_text = test_data.iloc[0]["input"]
prompt = f'test_data.iloc[0]["instruction"] {input_text}'

In [35]:
test_data.iloc[0]["output"]

'When a flying saucer lands in Washington, D.C., the Army quickly surrounds it. A humanoid (Michael Rennie) emerges, announcing that he has come in peace. When he unexpectedly opens a small device, he is shot by a nervous soldier. A tall robot emerges from the saucer and quickly disintegrates the soldiers\' weapons. Klaatu escapes and lodges at a boarding house as "Mr. Carpenter"'

In [33]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(base_model_id,
                                             device_map="auto",
                                             torch_dtype=None)

## Base Model

In [34]:


messages = [
    {
        "role": "system",
        "content": system
    },
    {
        "role": "user",
        "content": prompt
    }
]
response = generate_response(messages)
print(response)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Okay, I've analyzed the provided text. Here's a summary of the key events:

*   **Arrival and Initial Actions:** The Army surrounds the saucer, and a humanoid (Michael Rennie) emerges, announcing peace. He opens a small device, which is shot by a nervous soldier.
*   **The Alien's Arrival:** Klaatu, a tall, humanoid robot, emerges from the saucer and appears to be peaceful.
*   **The Alien's Motivation:** Klaatu's message is intended to be delivered to all world leaders simultaneously, which is impossible in the current political climate.
*   **The Alien's Actions:** Klaatu suggests that humans are irrational and have developed rockets and a rudimentary atomic power. He declares that if his message is ignored, Earth will be eliminated.
*   **Barnhardt's Arrival:** Barnhardt, a scientist, agrees to gather scientists from around the world.
*   **The Alien's Revelation:** Klaatu reveals his true identity as an interplanetary organization that created a police force of invincible robots, a

## Fine-tuned Model

In [38]:
model.load_adapter(adapter_model_id)

In [39]:

messages = [
    {
        "role": "system",
        "content": system
    },
    {
        "role": "user",
        "content": prompt
    }
]
response = generate_response(messages)
print(response)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


A humanoid (Michael Rennie) emerges, announcing that he has come in peace. A tall robot emerges from the saucer and disintegrates the soldiers' weapons. The alien orders the robot to stop, and the alien orders the robot to stop. The alien, Klaatu, is taken to Walter Reed Hospital. Klaatu escapes and lodges at a boarding house as "Mr. Carpenter", the name on the dry cleaner's tag on a suit he took.
